In [1]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

I0000 00:00:1776594589.475384 3071372 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776594589.501529 3071372 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776594590.097127 3071372 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Load Model

In [2]:
model_a_path = "model_a_initialized.h5"
model_b_path = "model_b_initialized.h5"

model_a = tf.keras.models.load_model(model_a_path, compile = False)
model_b = tf.keras.models.load_model(model_b_path, compile = False)

I0000 00:00:1776594591.029571 3071372 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22272 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
I0000 00:00:1776594591.029903 3071372 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22075 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:02:00.0, compute capability: 8.9


### LOAD DATASET: CIFAR-10

In [3]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Preprocessing
img_size = 64
x_test = tf.image.resize(x_test, (img_size, img_size)).numpy() / 255.0
y_test = tf.keras.utils.to_categorical(y_test, 10)

/home/master2/.conda/envs/xai/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


### Model Evaluation

In [4]:
dummy = tf.zeros((1, img_size, img_size, 3), dtype=tf.float32)

_ = model_a(dummy, training=False)

I0000 00:00:1776594593.560017 3071372 cuda_dnn.cc:461] Loaded cuDNN version 91002


In [9]:
y_prob = model_a.predict(x_test, batch_size = 32, verbose = 1)

y_pred = np.argmax(y_prob, axis = 1)
y_true = np.argmax(y_test, axis = 1)

accuracy = np.mean(y_pred == y_true)
print("Model A Accuracy: ", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
Model A Accuracy:  0.7466


In [10]:
y_prob = model_b.predict(x_test, batch_size = 32, verbose = 1)

y_pred = np.argmax(y_prob, axis = 1)
accuracy = np.mean(y_pred == y_true)
print("Model B Accuracy: ", accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step
Model B Accuracy:  0.8475
